In [1]:
from simulation_integration import *
import tensorflow as tf

# Enable GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

from simulation_integration import *
from PID_implementation import *

MODEL_PATH = "/Users/kadewidler/Desktop/widler_235585_original_unet_new_dataset_128px.h5"
PLATES_FOLDER = "/Users/kadewidler/Desktop/Y2B-2023-OT2_Twin/textures/_plates"


# Load model
model = simple_unet_model(128, 128, 3)
model.load_weights(MODEL_PATH)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=[f1])

pybullet build time: Jan 13 2026 15:41:40
/Users/kadewidler/Desktop/Y2B-2023-OT2_Twin/plant_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
2026-01-15 21:46:04.213121: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-01-15 21:46:04.213144: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-01-15 21:46:04.213149: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-01-15 21:46:04.213172: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-15 21:46:04.213186: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorF

In [ ]:
avg_positions = learn_plant_positions(PLATES_FOLDER, model)
zone_boundaries = create_zone_boundaries(avg_positions)

print("Zone boundaries:", zone_boundaries)


Analyzing 20 images to learn plant positions...
 4/17 [======>.......................] - ETA: 0s

2026-01-15 15:26:43.482837: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


17/17 [==============================] - 0s 24ms/step
✓ Learned positions from 13 images
  Average X-positions: ['1113.6', '1617.3', '2137.8', '2658.6', '3155.4']
Zone boundaries: [0, 1365.4764256047529, 1877.5832124869617, 2398.2159735624273, 2907.0142790981854, inf]


In [2]:
# Used to reload boundaries
import json
from pathlib import Path

ZONE_FILE = Path("/Users/kadewidler/Desktop/Y2B-2023-OT2_Twin/zone_boundaries.json")

with open(ZONE_FILE, "r") as f:
    zone_boundaries = np.array(json.load(f)["zone_boundaries"])

print("Loaded zone boundaries:", zone_boundaries)


Loaded zone boundaries: [   0.         1365.4764256  1877.58321249 2398.21597356 2907.0142791
           inf]


In [ ]:
from sim_class import Simulation
os.chdir("/Users/kadewidler/Desktop/Y2B-2023-OT2_Twin") # Used when directory changes
# Create simulation first 
sim = Simulation(num_agents=1, render=True)

# Get the matching plate image path
IMAGE_PATH = sim.get_plate_image()
print(f"Testing on: {IMAGE_PATH}")

# Process using that image
summary = process_with_simulation(sim, model, zone_boundaries)

sim.close()

Version = 4.1 Metal - 89.4
Vendor = Apple
Renderer = Apple M4
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
Testing on: textures/_plates/038_43-13-ROOT1-2023-08-08_control_pH7_-Fe+B_col0_04-Fish Eye Corrected.png

Processing plate: 038_43-13-ROOT1-2023-08-08_control_pH7_-Fe+B_col0_04-Fish Eye Corrected.png
Running CV pipeline...
 4/17 [======>.......................] - ETA: 0s

2026-01-15 21:46:20.304182: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


17/17 [==============================] - 1s 28ms/step
Detected 5/5 plants
Position 1: pixel ( 379,  995) -> sim (0.16062, 0.08214)
Position 2: pixel ( 913, 1054) -> sim (0.16375, 0.11051)
Position 3: pixel (1419,  994) -> sim (0.16057, 0.13740)
Position 4: pixel (2039, 1087) -> sim (0.16551, 0.17034)
Position 5: pixel (2444, 1127) -> sim (0.16763, 0.19186)
5 valid targets (sorted left-to-right)
Robot ready

Dispensing to 5 targets (left to right)...

Target 1/5: Position 1
Moving to (0.16062, 0.08214)
Arrived (error: 0.93mm, time: 5.49s)
Dispensed

Target 2/5: Position 2
Moving to (0.16375, 0.11051)
Arrived (error: 0.91mm, time: 5.28s)
Dispensed

Target 3/5: Position 3
Moving to (0.16057, 0.13740)
Arrived (error: 0.91mm, time: 5.33s)
Dispensed

Target 4/5: Position 4
Moving to (0.16551, 0.17034)
Arrived (error: 0.95mm, time: 5.37s)
Dispensed

Target 5/5: Position 5
Moving to (0.16763, 0.19186)
Arrived (error: 0.96mm, time: 5.26s)
Dispensed

Waiting for drops to settle...

Complete: 5/5

: 

In [4]:
import os
import random

textures_dir = "textures"
plates_dir = "textures/_plates"

texture_files = sorted([f for f in os.listdir(textures_dir) 
                        if not f.startswith('.') and not f.startswith('_')])
plate_files = sorted([f for f in os.listdir(plates_dir) 
                      if not f.startswith('.')])

print(f"Textures: {len(texture_files)}")
print(texture_files[:10])  # Show first 10

print(f"\nPlates: {len(plate_files)}")
print(plate_files[:10])  # Show first 10

print(f"\nMin available: {min(len(texture_files), len(plate_files))}")

Textures: 10
['01.png', '02.png', '03.png', '04.png', '05.png', '06.png', '07.png', '08.png', '09.png', '10.png']

Plates: 10
['030_43-2-ROOT1-2023-08-08_pvdCherry_OD001_Col0_05-Fish Eye Corrected.png', '031_43-6-ROOT1-2023-08-08_control_pH7_-Fe+B_f6h1_02-Fish Eye Corrected.png', '033_43-13-ROOT1-2023-08-08_pvd_OD01_Col0_02-Fish Eye Corrected.png', '033_43-14-ROOT1-2023-08-08_control_pH7_-Fe+B_col0_03-Fish Eye Corrected.png', '034_43-13-ROOT1-2023-08-08_control_pH7_-Fe+B_col0_02-Fish Eye Corrected.png', '035_43-17-ROOT1-2023-08-08_mock_pH5_+Fe_Col0_04-Fish Eye Corrected.png', '035_43-19-ROOT1-2023-08-08_pvd_OD01_f6h1_05-Fish Eye Corrected.png', '037_43-18-ROOT1-2023-08-08_mock_pH5_+Fe_Col0_04-Fish Eye Corrected.png', '038_43-13-ROOT1-2023-08-08_control_pH7_-Fe+B_col0_04-Fish Eye Corrected.png', '038_43-14-ROOT1-2023-08-08_pvdCherry_OD001_f6h1_04-Fish Eye Corrected.png']

Min available: 10


In [5]:
# Test if random is actually random
for i in range(5):
    idx = random.randint(0, len(plate_files) - 1)
    print(f"Random index {i}: {idx} -> {plate_files[idx]}")

Random index 0: 3 -> 033_43-14-ROOT1-2023-08-08_control_pH7_-Fe+B_col0_03-Fish Eye Corrected.png
Random index 1: 3 -> 033_43-14-ROOT1-2023-08-08_control_pH7_-Fe+B_col0_03-Fish Eye Corrected.png
Random index 2: 4 -> 034_43-13-ROOT1-2023-08-08_control_pH7_-Fe+B_col0_02-Fish Eye Corrected.png
Random index 3: 2 -> 033_43-13-ROOT1-2023-08-08_pvd_OD01_Col0_02-Fish Eye Corrected.png
Random index 4: 9 -> 038_43-14-ROOT1-2023-08-08_pvdCherry_OD001_f6h1_04-Fish Eye Corrected.png


In [ ]:
# Analyze Results (Kaggle Dataset)
df = pd.read_csv('dispensing_results.csv')

print("Dataset summary:")
print(f"  Total plants: {len(df)}")
print(f"  Successfully dispensed: {df['Dispensed'].sum()}")
print(f"  Success rate: {df['Dispensed'].sum()/len(df)*100:.1f}%")

successful = df[df['Converged'] == True]
print("\nPositioning accuracy:")
print(f"  Mean error: {successful['Error_mm'].mean():.2f} mm")
print(f"  Std error: {successful['Error_mm'].std():.2f} mm")
print(f"  Max error: {successful['Error_mm'].max():.2f} mm")

print("\nExecution time:")
print(f"  Mean time: {df['Time_sec'].mean():.2f} sec/plant")
print(f"  Total time: {df['Time_sec'].sum():.1f} sec")

Dataset summary:
  Total plants: 83
  Successfully dispensed: 83
  Success rate: 100.0%

Positioning accuracy:
  Mean error: 0.93 mm
  Std error: 0.06 mm
  Max error: 1.00 mm

Execution time:
  Mean time: 0.07 sec/plant
  Total time: 5.8 sec


: 